# 02 Tracking: run a line in one code, several codes, and a real machine

This notebook runs the seven scripts of `02_tracking/` in order with `%run`, so it runs exactly the same code as the command line (`python 02_tracking/NN_name.py`). The text here adds the explanation; each script's docstring says what it shows, what it needs and what it checks.

08 to 12 use small lattices built in Python by `simple_lattice.py`: a drift and FODO cells, written out as a master lattice (YAML, sections, layouts and a `.def`) exactly like a machine in laura-lattices. Their answers are known from linear optics, so each code is checked against a transfer matrix worked out from first principles. 13 and 14 track JFEL.

Everything here is tier A (pip only). 11, 12 and 14 add elegant when the simcodes image is there (see the README). The whole notebook takes a minute or two, most of it in 14.

## 08 Track a beam through a drift

The smallest tracking run: two markers 2 m apart, a `.def` with one line, run in Ocelot. The `files:` entry names the code, the starting beam (`input: particle_definition:`) and the line's first and last elements (`output:`). After `track()` the run directory holds the input deck SIMBA wrote and a beam file at each marker, `<name>.openpmd.hdf5`.

In a drift, $\sigma_x^2(L) = \langle x^2\rangle + 2L\langle xx'\rangle + L^2\langle x'^2\rangle$, from the starting beam's own moments, so the check needs no Twiss approximation.

In [ ]:
%run 08_track_drift.py

## 09 Track through FODO cells

Three FODO cells with a screen halfway along each drift. SIMBA fills the gaps between elements with drifts, so the lattice only lists quadrupoles and markers. The beam is deliberately mismatched, so its size beats along the line.

At every screen, the beam's second-moment matrix must be $\Sigma = M\Sigma_0M^T$ with $M$ the thick-lens transfer matrix. The script then reads the whole run back with `load_directory` and plots the beam size and emittance along $z$. Pass `master_lattice=` to `load_directory` (ISSUES.md #22).

In [ ]:
%run 09_track_fodo.py

In [ ]:
from IPython.display import Image, display
display(Image(filename="../output/09_track_fodo/envelope.png"))

## 10 Periodic optics of a FODO cell

`tracking: {periodic: true}` asks the code for the matched optics of the line, from its one-turn (here one-cell) map, instead of carrying the beam's own Twiss parameters through it. `framework["LINE"].ring_parameters()` then gives the tune and the periodic $\beta$, $\alpha$ at the start.

Check: $\cos\mu = \mathrm{Tr}(M)/2$ and $\beta = M_{12}/\sin\mu$. The thin-lens FODO formulas are printed alongside; they are up to 7% off for these 0.2 m quadrupoles.

In [ ]:
%run 10_twiss_without_particles.py

In [ ]:
display(Image(filename="../output/10_twiss_without_particles/twiss.png"))

## 11 Run the same line in different codes

`framework.change_Lattice_Code(line, code)` switches a line to another code without touching the lattice or the `.def`; `setSubDirectory` keeps each run separate. The same beam goes through 09's cells in Xsuite, Ocelot, Cheetah and MAD-X (and elegant), and the codes must agree particle by particle to $10^{-3}$ of the largest offset.

Two settings matter: MAD-X tracks a sliced (thin-lens) copy of the line, and Cheetah needs `cheetah_tracking_method: second_order` (in the `.def`'s `global:` block) to include chromatic focusing. The `XFAIL #23` line is a known SIMBA bug: `line.code` keeps the old name, although the tracking does switch.

In [ ]:
%run 11_change_code.py

## 12 Hand the beam from one code to the next

A `.def` with three lines in three codes. A line with no `input:` starts from the beam the previous line left at its `start_element`, and SIMBA converts the file format on the way. The chained run must match a single-code run of the whole channel, particle by particle, at each boundary.

In [ ]:
%run 12_start_to_end_handoff.py

## 13 A real machine: the JFEL linac

JFEL's `jfel_linac_codes.def` runs one line, Linac, from the saved injector beam (35 MeV, 250 pC) to the undulator entrance, 81 m on. The script switches it to Ocelot (or `--code ...`), turns off CSR, space charge and wakefields, and summarises the beam at the end of each section. The check is against the code-comparison table in JFEL's README.

To try another code from here: `%run 13_jfel_linac.py --code cheetah`.

In [ ]:
%run 13_jfel_linac.py

## 14 Compare codes on a real machine

13's line in Ocelot, Cheetah, MAD-X (and elegant), compared at every screen they share. Each pair's worst disagreement along the line is held to the ceiling SIMBA's own cross-code test uses. The codes are not expected to agree exactly: elegant compresses the bunch about 6% more (its RF model), and Ocelot grows $\varepsilon_{ny}$ by about 2% in the chicane.

`%run 14_compare_codes.py --codes ocelot xsuite` compares any set; pairs with no ceiling are printed but not checked.

In [ ]:
%run 14_compare_codes.py

In [ ]:
display(Image(filename="../output/14_compare_codes/compare.png"))